# <h1 style="text-align: center;">Fast Food Image Classification</h1>

# <h1 style="text-align: center;">Import Libraries</h1>

In [ ]:
#Import Os and Basis Libraries
import cv2
import os
import pandas as pd
import numpy as np
import seaborn as sns
from PIL import Image
import matplotlib.pyplot as plt 
#Matplot Images
import matplotlib.image as mpimg
# Tensflor and Keras Layer and Model and Optimize and Loss
import tensorflow as tf
import tensorflow_hub as hub
from tensorflow import keras
from keras import Sequential
from keras.layers import *
from tensorflow.keras.losses import BinaryCrossentropy
#Kernel Intilizer 
from sklearn.preprocessing import LabelEncoder
# import tensorflow_hub as hub
from tensorflow.keras.optimizers import Adam , Adamax
#PreTrained Model
from tensorflow.keras.applications import *
#Early Stopping
from tensorflow.keras.callbacks import EarlyStopping
from keras.preprocessing.image import ImageDataGenerator
# Warnings Remove 
import warnings 
warnings.filterwarnings("ignore")
# paellet 
palette = ["#606060FF", "#D6ED17FF"]

# <h1 style="text-align: center;">Load Data</h1>

In [ ]:
# Load Data and Make DataFrame
def L_Data(directory):
    filepath =[]
    label = []

    folds = os.listdir(directory)

    for fold in folds:
        f_path = os.path.join(directory , fold)

        imgs = os.listdir(f_path)

        for img in imgs:

            img_path = os.path.join(f_path , img)
            filepath.append(img_path)
            label.append(fold)

    #Concat data paths with labels
    file_path_series = pd.Series(filepath , name= 'filepath')
    Label_path_series = pd.Series(label , name = 'label')
    df_train = pd.concat([file_path_series ,Label_path_series ] , axis = 1)
    
    return df_train
# # Directory containing the "Train" folder
directory_T = "/kaggle/input/fast-food-classification-dataset/Fast Food Classification V2/Train"
# Directory containing the "Train" folder
directory_TE = "/kaggle/input/fast-food-classification-dataset/Fast Food Classification V2/Test"
# Train Data and Test Data 
tr_d = L_Data(directory_T)
te_d = L_Data(directory_TE)

In [ ]:
# Shape
print(f"The shape of The Train data is: {tr_d.shape}")
print(f"The shape of The Test data is: {te_d.shape}")

# <h1 style="text-align: center;">Train and Test Dataset Generator</h1>

In [ ]:
#Data_Dir Train And Test
test_dir = directory_TE
data_dir = "/kaggle/working/train_dataset"

# Image Size
IMAGE_SIZE = (256, 256)

print('Training Images:')
# creating the training dataset
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset='training',
    seed=123,
    image_size=IMAGE_SIZE,
    batch_size=32)

#Testing Augmented Data
print('Validation Images:')
validation_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir, 
    validation_split=0.2,
    subset='validation',
    seed=123,
    image_size=IMAGE_SIZE,
    batch_size=32)

# Create an ImageDataGenerator instance without augmentation
test_datagen = ImageDataGenerator(rescale=1.0/255.0)  

# Load test data using ImageDataGenerator
test_ds = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMAGE_SIZE,  
    batch_size=32,
    class_mode='categorical',  
    shuffle=False ,
)

# <h1 style="text-align: center;">Normalizing Pixel Value</h1>

In [ ]:
# Normalizing Pixel Values 
# Train Data 
train_ds = train_ds.map(lambda x, y: (x / 255.0, y))
# Val Data
validation_ds = validation_ds.map(lambda x, y: (x / 255.0, y))

# <h1 style="text-align: center;">Train and Test Label Count </h1>

In [ ]:
# Show COunt Function
def Count_S(df, palette):
    # Count the occurrences of each category in the 'label' column
    count = df['label'].value_counts()

    # Create a figure with two subplots
    fig, axs = plt.subplots(1, 2, figsize=(20, 6), facecolor='white')

    # Plot pie chart on the first subplot
    sns.set_palette(palette)
    axs[0].pie(count, labels=count.index, autopct='%1.1f%%', startangle=140)
    axs[0].set_title('Distribution of Categories')

    # Plot bar chart on the second subplot
    sns.barplot(x=count.index, y=count.values, ax=axs[1], palette=palette)
    axs[1].set_title('Count of Categories')

    # Adjust layout
    plt.tight_layout()

    # Show the plot
    plt.show()

# Define your custom palette
custom_palette = palette

In [ ]:
# Train Count
Count_S(tr_d, custom_palette)

In [ ]:
# Test Count
Count_S(te_d, custom_palette)

# <h1 style="text-align: center;">Train and Test Images</h1>

In [ ]:
def visualize_images(path, num_images=5):
    # Get a list of image filenames in the specified path
    image_filenames = os.listdir(path)
    
    # Limit the number of images to visualize if there are more than num_images
    num_images = min(num_images, len(image_filenames))
    
    # Create a figure and axis object to display images
    fig, axes = plt.subplots(1, num_images, figsize=(15, 3),facecolor='white')
    
    # Iterate over the selected images and display them
    for i, image_filename in enumerate(image_filenames[:num_images]):
        # Load the image using Matplotlib
        image_path = os.path.join(path, image_filename)
        image = mpimg.imread(image_path)
        
        # Display the image
        axes[i].imshow(image)
        axes[i].axis('off')  # Turn off axis
        axes[i].set_title(image_filename)  # Set image filename as title
    
    # Adjust layout and display the figure
    plt.tight_layout()
    plt.show()

# <h1 style="text-align: center;">Baked Potato</h1>

In [ ]:
# Specify the path containing the images to visualize
path_to_visualize = "/kaggle/input/fast-food-classification-dataset/Fast Food Classification V2/Train/Baked Potato"

# Visualize some images from the specified path
visualize_images(path_to_visualize, num_images=5)

# <h1 style="text-align: center;">Burger</h1>

In [ ]:
# Specify the path containing the images to visualize
path_to_visualize = "/kaggle/input/fast-food-classification-dataset/Fast Food Classification V2/Train/Burger"

# Visualize some images from the specified path
visualize_images(path_to_visualize, num_images=5)

# <h1 style="text-align: center;">Taco</h1>

In [ ]:
# Specify the path containing the images to visualize
path_to_visualize = "/kaggle/working/train_dataset/Taco"

# Visualize some images from the specified path
visualize_images(path_to_visualize, num_images=5)

# <h1 style="text-align: center;">Donut</h1>

In [ ]:
# Specify the path containing the images to visualize
path_to_visualize = "/kaggle/working/train_dataset/Donut"

# Visualize some images from the specified path
visualize_images(path_to_visualize, num_images=5)

# <h1 style="text-align: center;">Sandwich</h1>

In [ ]:
# Specify the path containing the images to visualize
path_to_visualize = "/kaggle/working/train_dataset/Sandwich"

# Visualize some images from the specified path
visualize_images(path_to_visualize, num_images=5)

# <h1 style="text-align: center;">Pizza</h1>

In [ ]:
# Specify the path containing the images to visualize
path_to_visualize = "/kaggle/working/train_dataset/Pizza"

# Visualize some images from the specified path
visualize_images(path_to_visualize, num_images=5)

# <h1 style="text-align: center;">Fries</h1>

In [ ]:
# Specify the path containing the images to visualize
path_to_visualize = "/kaggle/working/train_dataset/Fries"

# Visualize some images from the specified path
visualize_images(path_to_visualize, num_images=5)

# <h1 style="text-align: center;">Model Building | Transfer Learning | Bit Google Model</h1>

**Transfer learning is a machine learning technique where knowledge gained from training one model is applied to a different but related task. Instead of starting from scratch, a pre-trained model is used as a starting point. By leveraging features learned during the training of the pre-trained model, the new model can achieve better performance with less data and computation. This approach is particularly useful when working with limited labeled data or computational resources. Transfer learning involves fine-tuning the pre-trained model by adjusting its parameters to better suit the new task. This process allows for faster convergence and improved generalization to the new task. Overall, transfer learning accelerates the development of models for various tasks by capitalizing on the knowledge learned from previous tasks.**

In [ ]:
# Define the URL of the model
url = "https://tfhub.dev/google/bit/m-r50x1/1"

# Load the model from the URL
model_E = hub.KerasLayer(url)

# Set the model to be non-trainable
model_E.trainable = False

# <h1 style="text-align: center;">Build and Fit The Model</h1>

In [ ]:
def M_B(model_E ,EPO):
    # Define the name for your model
    model_name = "Food_Classification_Model"
    # Build the model
    model = Sequential(name=model_name)

    # Add the pre-trained DenseNet121_base 
    model.add(model_E)

    # Batch Normalization
    model.add(BatchNormalization())

    # Dropout 
    model.add(Dropout(0.2))

    # # Add a dense layer with 220 units and ReLU activation function
    # model.add(Dense(120, activation='relu'))

    # Add the output layer with 10 units and Softmax activation function
    model.add(Dense(10, activation='softmax'))

    # Compile
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    # Build the model
    model.build((None, 256, 256, 3))

    # Print model summary
    print(model.summary())
    
    #Early_Stopping
    early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

    #Fitting Model
    history = model.fit_generator(train_ds,
                            epochs= EPO,
                            validation_data = validation_ds,
                            callbacks = early_stopping)
    
    return history

In [ ]:
history = M_B(model_E, 6)

# <h1 style="text-align: center;">Validation Accuracy</h1>

In [ ]:
# Evaluate the model on the validation dataset
validation_loss, validation_accuracy = model.evaluate(validation_ds)

# Print the validation loss and accuracy
print("Validation Loss:", validation_loss)
print("Validation Accuracy:", validation_accuracy)

In [ ]:
# Get the epoch with the highest validation accuracy
best_epoch = history.history['val_accuracy'].index(max(history.history['val_accuracy'])) + 1

# Set the background style
plt.style.use('seaborn-darkgrid')

# Create a subplot with 1 row and 2 columns
fig, axs = plt.subplots(1, 2, figsize=(16, 5))

# Plot training and validation accuracy
axs[0].plot(history.history['accuracy'], label='Training Accuracy', color='blue')
axs[0].plot(history.history['val_accuracy'], label='Validation Accuracy', color='red')
axs[0].scatter(best_epoch - 1, history.history['val_accuracy'][best_epoch - 1], color='green', label=f'Best Epoch: {best_epoch}')
axs[0].set_xlabel('Epoch')
axs[0].set_ylabel('Accuracy')
axs[0].set_title('Training and Validation Accuracy')
axs[0].legend()

# Plot training and validation loss
axs[1].plot(history.history['loss'], label='Training Loss', color='blue')
axs[1].plot(history.history['val_loss'], label='Validation Loss', color='red')
axs[1].scatter(best_epoch - 1, history.history['val_loss'][best_epoch - 1], color='green',label=f'Best Epoch: {best_epoch}')
axs[1].set_xlabel('Epoch')
axs[1].set_ylabel('Loss')
axs[1].set_title('Training and Validation Loss')
axs[1].legend()

plt.tight_layout()
plt.show()